In [ ]:
from google.colab import userdata, drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
!pip install -q -U crewai
!pip install -q -U duckduckgo-search
!pip install -q -U crewai[tools]

import os
from crewai import Agent, Task, Crew, Process, LLM
import google.generativeai as genai

from crewai.tools import BaseTool

# make sure to update with your Gemini API key
gemini_api_key = "xxxxxxxxxxxxxxxxxxxx"

# Configure the API key
genai.configure(api_key=gemini_api_key)
#drive.mount('/content/drive')
file_path = '/content/drive/My Drive/Colab Notebooks/xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx' # Update with the correct path of your file

try:
    with open(file_path, 'r') as f:
        csv_string = f.read()
    print("File content read successfully!")
    print(csv_string[:500]) # Print the first 500 characters to show the content
except FileNotFoundError:
    print(f"File not found at {file_path}. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
# Use a potentially more stable or capable model
# If you are rate limited, pick another model
gemini_llm = LLM(
    model= 'gemini/gemini-2.0-flash-lite', ###'gemini/gemini-2.5-pro', ###'gemini-1.5-flash-latest', ###'gemini/gemini-2.0-flash-lite', ###'gemini/gemini-2.5-pro',
    api_key=gemini_api_key,
    temperature=0.0  # Lower temperature for more consistent results.
)

# If issues persist, consider simplifying the tasks.

In [ ]:
print(csv_string[:500])
#

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
import pandas as pd

# Define a custom tool to read the CSV string
class CsvStringReaderTool(BaseTool):
    name: str = "CSV String Reader"
    description: str = "Reads the financial data from the provided CSV string."
    csv_data: str

    def _run(self, query: str = None) -> str:
        """Reads the CSV data."""
        return self.csv_data

# Instantiate the custom tool with your csv_string
csv_reader_tool = CsvStringReaderTool(csv_data=csv_string)

# Define one of your agents
data_analyst = Agent(
    role='=========== Define this agents role ==============',
    goal='=============== What is the goal ===============',
    backstory='============================== Give your agent some context ===================================',
    verbose=True,
    allow_delegation=False,
    llm=gemini_llm, # Use the configured Gemini LLM
    tools=[csv_reader_tool] # Add the custom tool to the agent
)

# Define the task for the agent
analyze_data_task = Task(
    description=f"""

  provide a clear description of wha ttihs agent needs to do.  look at Assignment5 for example/

""",
    expected_output='A summary summarizing the key findings from xxxxxxxxxxxxxxxxxxxxxxx.',
    agent=data_analyst
)

In [ ]:
# Define another  agent
stock_predictor = Agent(
    role='=========== Define this agents role ==============',
    goal='=============== What is the goal ===============',
    backstory='============================== Give your agent some context ===================================',
    verbose=True,
    allow_delegation=False,
    llm=gemini_llm # Use the configured Gemini LLM
)

print(csv_string[0:500])
# Define the task for the stock price prediction agent
predict_stock_price_task = Task(
    description=f"""

  provide a clear description of wha ttihs agent needs to do.  look at Assignment5 for example/

""",
    expected_output='A summary summarizing the key findings from xxxxxxxxxxxxxxxxxxxxxxx.',
    agent=stock_predictor,
    context=[analyze_data_task] # Add the analysis task as context
)


### update the names here...

# Create a crew with both agents and tasks
financial_analysis_and_prediction_crew = Crew(
    agents=[data_analyst, stock_predictor],
    tasks=[analyze_data_task, predict_stock_price_task],
    process=Process.hierarchical, # Use hierarchical process for multiple agents
    manager_llm=gemini_llm # Specify the manager LLM
)

# Kick off the crew's work
print("## Starting the xxxxxxxxxxx Crew")
result = financial_analysis_and_prediction_crew.kickoff()

# Print the results
print("## Final Report:")
print(result)